In [3]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

2026-06-19 15:45:46.310658542 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [4]:
v1.dot(dv)

np.float64(0.3233238799303238)

In [5]:
v2.dot(dv)

np.float64(0.019730422395141473)

In [6]:
q = 'How does approximate nearest neighbor search work?'
v = embed.encode(q)

Q1. Embedding a query
The embedder returns a vector of 384 numbers. What's the first value (v[0])?
-0.31
-0.02
0.12
0.44

In [7]:
v[0]

np.float64(-0.02058203437252893)

In [8]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [9]:
len(documents)

72

Q2. Cosine similarity

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

0.07
0.37
0.68
0.92

In [10]:
result = [x for x in documents if x["filename"]=="02-vector-search/lessons/07-sqlitesearch-vector.md"]
result

[{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done

In [17]:
content = result[0]['content']
c = embed.encode(content)

In [ ]:
c.dot(v)

np.float64(0.36107026789538205)

Q3. Chunking and search by hand

Which file does the highest-scoring chunk belong to (its filename)?

02-vector-search/lessons/03-embeddings-dataset.md

02-vector-search/lessons/06-rag-vector.md

02-vector-search/lessons/07-sqlitesearch-vector.md

02-vector-search/lessons/09-onnx-embedder.md

In [13]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [16]:
chunks[2]

{'start': 2000,
 'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline agen

In [21]:
len(chunks)

295

In [49]:
x = 0
results=[]

for i in chunks:
    if x >= len(chunks):
        break
    else:
        content = chunks[x]['content']
        filename = chunks[x]['filename']
        c = embed.encode(content)
        score = c.dot(v)
        results.append((float(score), filename))
        x +=1

results.sort(reverse=True)
results[0][1]        

'02-vector-search/lessons/07-sqlitesearch-vector.md'